# Workshop - Mensageria com Listas, Pub/Sub e Streams

<img src="https://cdn.sanity.io/images/sy1jschh/production/b4cdf336c42284d92034fffb2b14d8a79518ae82-621x456.svg" alt="Redis" width="120">

Neste notebook vamos explorar três formas de trocar mensagens entre serviços com o Redis: filas com Listas, Pub/Sub e Streams. Os exemplos usam um sistema acadêmico, com eventos de cadastro de alunos e de lançamento de notas.

Para uma experiência completa, recomendamos abrir o **Redis Insight** (App ou Web) em paralelo, para visualizar os dados sendo criados e consultados em tempo real.

https://redis.io/insight/

## Objetivos deste notebook

- Implementar uma fila FIFO com Listas para processar cadastros de alunos
- Publicar lançamentos de notas com Pub/Sub e acompanhar a entrega pelo Redis Insight
- Registrar lançamentos de notas em uma Stream e controlar o seu tamanho
- Ler a Stream com `XREAD`, `XRANGE` e `XREVRANGE`
- Distribuir o processamento com Consumer Groups, confirmando e recuperando mensagens pendentes

## Pré-requisitos

Cadastre os secrets abaixo no Colab (ícone de 🔑 **Secrets** no menu lateral esquerdo), com acesso liberado para este notebook:

- `REDIS_HOST`
- `REDIS_PORT`
- `REDIS_PASSWORD`

Para a Parte Dois (Pub/Sub), também é necessário ter o **Redis Insight** conectado ao mesmo banco.

## Setup do Ambiente

In [ ]:
!pip install -q redis

## Setup da Conexão

In [ ]:
from google.colab import userdata

REDIS_HOST = userdata.get("REDIS_HOST")
REDIS_PORT = int(userdata.get("REDIS_PORT"))
REDIS_PASSWORD = userdata.get("REDIS_PASSWORD")

## Teste de Conexão

In [ ]:
import redis

r = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    decode_responses=True,
)

print("Conexão ok:", r.ping())

# Parte Um - Filas com Listas

Uma fila organiza mensagens em ordem de chegada: a primeira mensagem inserida é a primeira a ser processada (FIFO — *First In, First Out*). Esse padrão desacopla quem produz o trabalho de quem o executa.

No Redis, uma fila é implementada com uma **Lista**: os produtores inserem itens em uma extremidade (`LPUSH`) e os consumidores removem da outra (`RPOP`).

Neste exemplo, o sistema de matrículas publica cadastros de novos alunos em uma fila, e um serviço de integração os processa na ordem em que chegaram.

## Publicando cadastros na fila (LPUSH)

In [ ]:
import json

FILA_CADASTROS = "academico:fila:cadastros"
r.delete(FILA_CADASTROS)

cadastros = [
    {"matricula": "2026001", "nome": "Ana Souza", "curso": "Engenharia de Software"},
    {"matricula": "2026002", "nome": "Bruno Lima", "curso": "Ciência de Dados"},
    {"matricula": "2026003", "nome": "Carla Mendes", "curso": "Sistemas de Informação"},
    {"matricula": "2026004", "nome": "Diego Rocha", "curso": "Engenharia de Software"},
]

for cadastro in cadastros:
    r.lpush(FILA_CADASTROS, json.dumps(cadastro, ensure_ascii=False))

print("Itens na fila:", r.llen(FILA_CADASTROS))
for item in r.lrange(FILA_CADASTROS, 0, -1):
    print(item)

`LPUSH` insere cada cadastro no início da lista, por isso o `LRANGE` mostra o cadastro mais recente primeiro. `LLEN` retorna a quantidade de itens na fila.

## Consultando a fila sem consumir (peek)

Antes de consumir, é possível inspecionar a fila sem remover nenhum item. Como os itens são consumidos pelo fim da lista, o próximo a ser processado é o último elemento.

In [ ]:
proximo = r.lrange(FILA_CADASTROS, -1, -1)
print("Próximo a ser processado:", proximo)

print("Fila completa:")
for item in r.lrange(FILA_CADASTROS, 0, -1):
    print(item)

## Consumindo um cadastro (RPOP)

`RPOP` remove e retorna o último elemento da lista, ou seja, o cadastro mais antigo. Execute a célula mais de uma vez para acompanhar a fila sendo consumida.

In [ ]:
item = r.rpop(FILA_CADASTROS)

if item:
    print("Cadastro consumido:", json.loads(item))
else:
    print("A fila está vazia.")

print("Itens restantes na fila:", r.llen(FILA_CADASTROS))

## Processando a fila até esvaziar

Um consumidor real fica em um loop: retira o próximo item, processa e repete até não haver mais trabalho.

In [ ]:
import time

def processar_cadastro(cadastro):
    time.sleep(0.5)
    print(f"Cadastro processado: {cadastro['matricula']} - {cadastro['nome']}")

while True:
    item = r.rpop(FILA_CADASTROS)
    if item is None:
        print("Fila vazia. Todos os cadastros foram processados.")
        break
    processar_cadastro(json.loads(item))

# Parte Dois - Pub/Sub

No padrão **Publish/Subscribe**, um produtor publica mensagens em um canal e todos os assinantes daquele canal recebem a mesma mensagem ao mesmo tempo. Diferente da fila, onde cada item é processado por um único consumidor, o Pub/Sub entrega a mensagem para todos os interessados.

O Pub/Sub é *fire-and-forget*: as mensagens não ficam armazenadas. Um assinante que não estiver conectado no momento da publicação não recebe a mensagem.

Neste exemplo, a secretaria publica cada lançamento de nota no canal `academico:notas`, para que sistemas como o portal do aluno e o serviço de notificações sejam avisados em tempo real.

## Assinando o canal no Redis Insight

1. Abra o Redis Insight e conecte-se ao mesmo banco utilizado neste notebook.
2. No menu lateral, abra a opção **Pub/Sub**.

![](https://github.com/gacerioni/redis-workshop-mensageria-pubsub-streams/blob/master/static/pubsub_menu.png?raw=true)

3. Mantenha o padrão de canal `*` (ou informe `academico:notas`) e clique em **Subscribe**.
4. Volte ao notebook e execute a célula abaixo. O lançamento publicado aparece na lista de mensagens do Redis Insight.

## Publicando um lançamento de nota (PUBLISH)

In [ ]:
CANAL_NOTAS = "academico:notas"

lancamento = {"matricula": "2026001", "disciplina": "Estruturas de Dados", "nota": 8.5}

assinantes = r.publish(CANAL_NOTAS, json.dumps(lancamento, ensure_ascii=False))
print(f"Lançamento publicado em '{CANAL_NOTAS}' para {assinantes} assinante(s)")

`PUBLISH` retorna a quantidade de assinantes que receberam a mensagem. Para comparar, clique em **Unsubscribe** no Redis Insight e publique novamente: o retorno será `0` e a mensagem será descartada, sem ficar disponível para quem se inscrever depois.

Também é possível publicar direto pelo Redis Insight, informando o canal e a mensagem na mesma tela.

Quando é preciso garantir a entrega e manter o histórico das mensagens, a estrutura indicada é a **Stream**, apresentada a seguir.

# Parte Três - Streams

Uma **Stream** é um log de eventos *append-only*: cada nova entrada é adicionada ao final, recebe um ID único baseado em tempo e fica armazenada no Redis. Isso permite reler o histórico, consumir a partir de qualquer ponto e confirmar o processamento de cada mensagem.

Cada entrada é um conjunto de pares campo/valor, semelhante a um `HASH`, e entradas da mesma Stream podem ter campos diferentes.

Neste exemplo, cada lançamento de nota é registrado na Stream `academico:stream:notas`.

## Registrando lançamentos de notas (XADD)

In [ ]:
STREAM_NOTAS = "academico:stream:notas"
r.delete(STREAM_NOTAS)

lancamentos = [
    {"matricula": "2026001", "disciplina": "Estruturas de Dados", "nota": "8.5"},
    {"matricula": "2026002", "disciplina": "Estruturas de Dados", "nota": "6.0"},
    {"matricula": "2026003", "disciplina": "Banco de Dados", "nota": "9.0"},
    {"matricula": "2026002", "disciplina": "Banco de Dados", "nota": "4.5", "observacao": "Aluno em recuperação"},
    {"matricula": "2026004", "disciplina": "Cálculo I", "nota": "7.0", "avaliacao": "Segunda chamada"},
]

for lancamento in lancamentos:
    entry_id = r.xadd(STREAM_NOTAS, lancamento)
    print(entry_id, lancamento)

Repare que os dois últimos lançamentos têm campos extras (`observacao` e `avaliacao`). A Stream não exige um schema fixo entre as entradas.

## Limitando o tamanho da Stream (XTRIM e MAXLEN)

Como a Stream mantém todas as entradas, é importante controlar o seu tamanho. `XTRIM` remove as entradas mais antigas até a Stream ficar com o limite definido. O mesmo limite pode ser aplicado na escrita, com o parâmetro `maxlen` do `XADD`.

Com `approximate=False`, o limite é exato. Em produção, `approximate=True` é mais eficiente, porque o Redis remove entradas em blocos e o tamanho pode ficar levemente acima do limite.

In [ ]:
print("Tamanho antes:", r.xlen(STREAM_NOTAS))

r.xtrim(STREAM_NOTAS, maxlen=3, approximate=False)

print("Tamanho depois:", r.xlen(STREAM_NOTAS))
for entry_id, campos in r.xrange(STREAM_NOTAS, "-", "+"):
    print(entry_id, campos)

## Lendo a Stream

A partir daqui, a Stream contém apenas os 3 lançamentos mais recentes.

### XREAD

`XREAD` lê entradas a partir de um ID. Os parâmetros usados abaixo são:

- **`"0"`:** lê desde o início da Stream
- **`count=2`:** retorna no máximo 2 entradas
- **`block=300`:** se a Stream estiver vazia, aguarda até 300 ms por novas entradas

In [ ]:
resultado = r.xread({STREAM_NOTAS: "0"}, count=2, block=300)

for stream, entradas in resultado:
    for entry_id, campos in entradas:
        print(entry_id, campos)

### XRANGE

`XRANGE` retorna as entradas dentro de um intervalo de IDs. Os símbolos `-` e `+` representam o menor e o maior ID possíveis, ou seja, a Stream inteira.

In [ ]:
for entry_id, campos in r.xrange(STREAM_NOTAS, "-", "+"):
    print(entry_id, campos)

### XREVRANGE

`XREVRANGE` funciona como o `XRANGE`, mas em ordem reversa, da entrada mais recente para a mais antiga. Por isso o intervalo é informado como `+` e `-`.

In [ ]:
for entry_id, campos in r.xrevrange(STREAM_NOTAS, "+", "-"):
    print(entry_id, campos)

## Consumer Groups

Um **Consumer Group** permite que vários consumidores dividam o processamento de uma mesma Stream: cada entrada é entregue a apenas um consumidor do grupo. Grupos diferentes, por outro lado, recebem todas as entradas de forma independente. Assim, o serviço de boletins e o serviço de notificações podem processar os mesmos lançamentos, cada um no seu próprio grupo.

O Redis controla, para cada grupo, quais entradas já foram entregues e quais ainda aguardam confirmação.

### Criando o grupo (XGROUP CREATE)

- **`id="0"`:** o grupo começa a ler desde a primeira entrada da Stream
- **`mkstream=True`:** cria a Stream caso ela ainda não exista

Depois de criar o grupo, abra a chave `academico:stream:notas` no Redis Insight para visualizá-lo. Para recomeçar o exercício, execute novamente a célula do `XADD`, que recria a Stream sem nenhum grupo.

In [ ]:
GRUPO_BOLETIM = "servico-boletim"

r.xgroup_create(STREAM_NOTAS, GRUPO_BOLETIM, id="0", mkstream=True)
print(f"Grupo '{GRUPO_BOLETIM}' criado na Stream '{STREAM_NOTAS}'")

### Lendo mensagens pelo grupo (XREADGROUP)

`XREADGROUP` lê entradas em nome de um consumidor do grupo. O ID especial `>` indica que o consumidor quer apenas entradas que ainda não foram entregues a nenhum outro consumidor do grupo.

O mesmo comando aceita várias Streams de uma vez, no formato `{stream_a: ">", stream_b: ">"}`.

In [ ]:
resultado = r.xreadgroup(GRUPO_BOLETIM, "consumidor-1", {STREAM_NOTAS: ">"}, count=2, block=3000)

mensagens_consumidor1 = resultado[0][1] if resultado else []

print("Entradas recebidas por consumidor-1:")
for entry_id, campos in mensagens_consumidor1:
    print(entry_id, campos)

Um segundo consumidor entra no grupo e recebe apenas o que ainda não foi entregue:

In [ ]:
resultado = r.xreadgroup(GRUPO_BOLETIM, "consumidor-2", {STREAM_NOTAS: ">"}, count=2, block=3000)

mensagens_consumidor2 = resultado[0][1] if resultado else []

print("Entradas recebidas por consumidor-2:")
for entry_id, campos in mensagens_consumidor2:
    print(entry_id, campos)

### Consultando mensagens pendentes (XPENDING)

Toda entrada entregue a um consumidor fica na **PEL** (*Pending Entries List*) do grupo até ser confirmada. `XPENDING` mostra quantas entradas estão pendentes e com quais consumidores.

In [ ]:
pendentes = r.xpending(STREAM_NOTAS, GRUPO_BOLETIM)
print("Mensagens pendentes:", pendentes)

### Confirmando o processamento (XACK)

Depois de processar uma entrada, o consumidor confirma com `XACK`, e a entrada sai da lista de pendentes. Abaixo, o `consumidor-1` confirma a primeira entrada que recebeu.

In [ ]:
if mensagens_consumidor1:
    entry_id, campos = mensagens_consumidor1[0]
    r.xack(STREAM_NOTAS, GRUPO_BOLETIM, entry_id)
    print(f"Entrada {entry_id} confirmada pelo consumidor-1")
else:
    print("O consumidor-1 não tem entradas para confirmar.")

print("Mensagens pendentes:", r.xpending(STREAM_NOTAS, GRUPO_BOLETIM))

### Recuperando mensagens pendentes (XAUTOCLAIM)

Se um consumidor falhar depois de receber uma entrada, ela fica pendente indefinidamente. `XAUTOCLAIM` localiza as entradas pendentes há mais tempo do que o limite informado e transfere a posse delas para outro consumidor, em um único comando.

O limite abaixo é de 6 segundos (`min_idle_time=6000`), para que o efeito seja visível no workshop; em produção esse valor costuma ser de minutos. Aguarde alguns segundos após o `XREADGROUP` antes de executar a célula.

In [ ]:
_, entradas_recuperadas, _ = r.xautoclaim(
    STREAM_NOTAS, GRUPO_BOLETIM, "consumidor-2",
    min_idle_time=6000,
    start_id="0",
)

print("Entradas recuperadas pelo consumidor-2:")
for entry_id, campos in entradas_recuperadas:
    print(entry_id, campos)
    r.xack(STREAM_NOTAS, GRUPO_BOLETIM, entry_id)

print("Mensagens pendentes:", r.xpending(STREAM_NOTAS, GRUPO_BOLETIM))

# Recapitulando

- **Filas:** `LPUSH` e `RPOP` em uma Lista para processar cadastros de alunos em ordem de chegada
- **Pub/Sub:** `PUBLISH` para avisar todos os assinantes de um lançamento de nota, sem armazenar a mensagem
- **Streams:** `XADD`, `XTRIM`, `XREAD`, `XRANGE` e `XREVRANGE` para registrar e consultar o histórico de lançamentos
- **Consumer Groups:** `XGROUP CREATE`, `XREADGROUP`, `XPENDING`, `XACK` e `XAUTOCLAIM` para dividir o processamento com garantia de entrega

Este notebook faz parte de uma série de treinamento em Redis. Os outros módulos são:

- `search.ipynb` — Cache, JSON, índices e busca (textual e vetorial) com Redis Search
- `streams.ipynb` — Filas, Pub/Sub e Streams para mensageria
- `agent_memory.ipynb` — Memória de curto e longo prazo para agentes de IA
- `langcache.ipynb` — Cache semântico de respostas de LLM com LangCache